<a href="https://colab.research.google.com/github/carloscastillo-hue/reto-inteligencia-operativa/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import math
import sqlite3
import pandas as pd

# 1. Cargar el archivo CSV subido a Colab
df_casos = pd.read_csv('siniestros_movilidad.csv')
print('--- DATOS CARGADOS EXITOSAMENTE ---')
display(df_casos.head())

# 2. Crear base de datos SQL en memoria (simula tus tablas operativas)
conn = sqlite3.connect(':memory:')
df_casos.to_sql('casos_movilidad', conn, index=False)

# Tabla simulada de aliados disponibles en Bogotá
df_aliados = pd.DataFrame({
    'id_aliado': ['AL-01', 'AL-02', 'AL-03', 'AL-04'],
    'nombre_aliado': [
        'Grúas Express Norte',
        'Asistencia Total 24/7',
        'MovilSur Aliados',
        'Taller Rápido Centro',
    ],
    'tipo_servicio': ['grua', 'grua', 'grua', 'carro_taller'],
    'lat_aliado': [4.7180, 4.6600, 4.5800, 4.6550],
    'lon_aliado': [-74.0650, -74.0550, -74.1050, -74.0780],
    'disponible': [1, 1, 1, 1],
})
df_aliados.to_sql('aliados', conn, index=False)

# 3. Consulta SQL de negocio: Filtrar casos con póliza vigente y marcar alertas de alto costo (HITL)
query = """
SELECT
    id_caso,
    placa,
    tipo_asistencia,
    zona,
    costo_estimado,
    CASE
        WHEN costo_estimado > 250000 THEN 'SI - Validación Humana (HITL)'
        ELSE 'NO - Asignación Automática'
    END AS requiere_aprobacion_humana
FROM casos_movilidad
WHERE estado_poliza = 'Vigente'
ORDER BY costo_estimado DESC
"""
df_procesados = pd.read_sql_query(query, conn)
print('\n--- REGLAS DE NEGOCIO Y GOBIERNO (HITL) APLICADAS ---')
display(df_procesados)


# 4. Función Geoespacial (Haversine) para encontrar el aliado más cercano
def distancia_km(lat1, lon1, lat2, lon2):
  r = 6371.0
  dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
  a = (
      math.sin(dlat / 2) ** 2
      + math.cos(math.radians(lat1))
      * math.cos(math.radians(lat2))
      * math.sin(dlon / 2) ** 2
  )
  return round(r * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))), 2)


# Ejemplo: Asignar grúa más cercana al primer caso (CASO-001)
caso = df_casos.iloc[0]
gruas = df_aliados[df_aliados['tipo_servicio'] == 'grua'].copy()
gruas['distancia_km'] = gruas.apply(
    lambda row: distancia_km(
        caso['lat_cliente'],
        caso['lon_cliente'],
        row['lat_aliado'],
        row['lon_aliado'],
    ),
    axis=1,
)
mejor_aliado = gruas.sort_values('distancia_km').iloc[0]

print(
    f"\n--- RESULTADO ASIGNACIÓN GEOESPACIAL PARA {caso['id_caso']}"
    f" ({caso['placa']}) ---"
)
print(
    f"Aliado recomendado: {mejor_aliado['nombre_aliado']} a"
    f" {mejor_aliado['distancia_km']} km de distancia."
)

--- DATOS CARGADOS EXITOSAMENTE ---


,id_caso,fecha_hora,placa,tipo_asistencia,lat_cliente,lon_cliente,zona,descripcion_cliente,estado_poliza,costo_estimado
0,CASO-001,2026-11-23 07:15,ABC123,grua,4.7110,-74.0721,Norte,El carro se apagó en la Autopista Norte con 13...,Vigente,145000
1,CASO-002,2026-11-23 07:30,XYZ987,carro_taller,4.6500,-74.0836,Occidente,Me pinché saliendo por la Calle 26 y no tengo ...,Vigente,85000
2,CASO-003,2026-11-23 07:45,LMN456,grua,4.6097,-74.0817,Centro,"Choque simple en la Carrera 10, el radiador qu...",Vencida,160000
3,CASO-004,2026-11-23 08:00,QWE321,cerrajeria,4.6768,-74.0483,Chapinero,Dejé las llaves adentro del baúl en el parquea...,Vigente,75000
4,CASO-005,2026-11-23 08:10,RTY654,grua,4.5709,-74.1130,Sur,"Falla mecánica en la Avenida Boyacá, humo en e...",Vigente,320000



--- REGLAS DE NEGOCIO Y GOBIERNO (HITL) APLICADAS ---


,id_caso,placa,tipo_asistencia,zona,costo_estimado,requiere_aprobacion_humana
0,CASO-005,RTY654,grua,Sur,320000,SI - Validación Humana (HITL)
1,CASO-008,DFG753,grua,Norte,290000,SI - Validación Humana (HITL)
2,CASO-001,ABC123,grua,Norte,145000,NO - Asignación Automática
3,CASO-002,XYZ987,carro_taller,Occidente,85000,NO - Asignación Automática
4,CASO-006,UIO789,carro_taller,Norte,80000,NO - Asignación Automática
5,CASO-004,QWE321,cerrajeria,Chapinero,75000,NO - Asignación Automática



--- RESULTADO ASIGNACIÓN GEOESPACIAL PARA CASO-001 (ABC123) ---
Aliado recomendado: Grúas Express Norte a 1.11 km de distancia.
